# Análise Exploratória e de Séries Temporais Multivariadas — Compressor Industrial (CBM 1º Estágio)

Este notebook realiza uma investigação abrangente de **múltiplas séries temporais provenientes de sensores e indicadores termodinâmicos de monitoramento baseado em condição (*Condition-Based Monitoring — CBM*)** do **1º Estágio de um Compressor Centrífugo Industrial** (`/home/jalcantara/DeepLearning2026-3/data/compressor.csv`).

---

## Sumário Analítico

1. **Configuração do Ambiente, Carregamento e Inspeção Estrutural**
2. **Dicionário Físico dos Sensores (Grandezas *Actual*, *Expected* e *Deviation*)**
3. **Auditoria de Qualidade de Dados e Diagnóstico do Historiador Industrial**
   - Valores ausentes e padrões de co-ocorrência (*Split-Row Sampling* `28s + 2s = 30s`)
   - Regularização temporal na grade de `30s` e análise de lacunas (*Deadband Compression*)
   - Validação matemática dos desvios CBM e reconstrução analítica da variável 100% nula (`Poly Eff Dev`)
   - Detecção de paradas operacionais (`zeros`) e transientes extremos de partida/parada
4. **Visualização Multivariada das Séries Temporais (Macro de 6 Meses e Micro em Transientes)**
   - Comparação *Actual vs. Expected* nas 4 grandezas termodinâmicas
   - Evolução conjunta dos 4 desvios percentuais e ciclos de degradação/recuperação
   - Zoom de alta resolução (`30s`) em evento de parada, partida e *glitch* numérico
5. **Distribuições Estatísticas, Sazonalidade Multi-Escala e Estacionariedade**
   - Estatísticas robustas, assimetria, curtose, histogramas/KDE e boxplots padronizados
   - Sazonalidade horária (ciclo térmico diurno), semanal e mensal (mapas de calor Mês × Hora)
   - Decomposição Tendência-Sazonalidade-Resíduo (STL) e testes formais de estacionariedade (ADF e KPSS)
6. **Dinâmica Temporal: Memória, Defasagem (Lead-Lag) e Domínio da Frequência**
   - Autocorrelação (ACF) e Autocorrelação Parcial (PACF) em curto e médio prazo
   - Correlação Cruzada (*Cross-Correlation* / *Lead-Lag*) entre potência, pressão e eficiência
   - Densidade Espectral de Potência (Método de Welch / FFT) para identificação de periodicidades
7. **Análise Multivariada, Mapas Termodinâmicos, PCA (MSPC) e Regimes Operacionais**
   - Matrizes de correlação de Pearson e Spearman + agrupamento hierárquico de sensores
   - Mapas característicos de performance do compressor ($r_p \times H_p$, Potência Real vs. Esperada)
   - Redução de dimensionalidade via PCA e cartas de controle multivariadas ($T^2$ de Hotelling e erro de reconstrução SPE/$Q$)
   - Descoberta não supervisionada de regimes operacionais via *Clustering* (*K-Means*) na linha do tempo
8. **Detecção de Anomalias Multivariadas, Mudanças Estruturais (CUSUM) e Preparação para Deep Learning**
   - Comparação entre detecção univariada (Z-Score Robusto / MAD) e multivariada (*Isolation Forest* + PCA-SPE)
   - Monitoramento de deriva estrutural via CUSUM e volatilidade móvel
   - Pipeline livre de *data leakage* (divisão cronológica, normalização e janelamento 3D) pronto para redes neurais (LSTM, GRU, TCN, Transformer / Autoencoders)

## 1. Configuração do Ambiente e Importações

Utilizamos bibliotecas científicas padrão do ecossistema Python (`numpy`, `pandas`, `matplotlib`, `seaborn`, `scipy`, `scikit-learn`) e, quando disponível no ambiente, `statsmodels` (com *fallbacks* vetoriais equivalentes caso `statsmodels` não esteja instalado no kernel).

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from scipy import stats, signal
from scipy.cluster import hierarchy
from scipy.spatial.distance import squareform

from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest
from sklearn.metrics import silhouette_score

try:
    from IPython.display import display
except ImportError:
    display = print

try:
    from statsmodels.tsa.seasonal import STL
    from statsmodels.tsa.stattools import adfuller, kpss, acf, pacf
    HAS_STATSMODELS = True
except ImportError:
    HAS_STATSMODELS = False

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# Estilo visual consistente para relatórios técnicos
sns.set_theme(style="whitegrid", context="notebook", palette="deep")
plt.rcParams.update({
    "figure.figsize": (10, 5),
    "figure.dpi": 110,
    "axes.titlesize": 11,
    "axes.titleweight": "bold",
    "axes.labelsize": 10,
    "legend.fontsize": 9,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
})

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Diretório de saída para salvar os gráficos da EDA em formato PDF individual
REPO_ROOT = Path("/home/jalcantara/DeepLearning2026-3").resolve()
EDA_RESULTS_DIR = REPO_ROOT / "results" / "eda"
EDA_SERIES_DIR = EDA_RESULTS_DIR / "series_temporais"

EDA_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
EDA_SERIES_DIR.mkdir(parents=True, exist_ok=True)

def save_eda_fig(fig, filename: str, subfolder: Path = EDA_RESULTS_DIR):
    """Salva a figura individualmente em PDF no diretório especificado."""
    if not filename.endswith(".pdf"):
        filename += ".pdf"
    subfolder.mkdir(parents=True, exist_ok=True)
    out_path = subfolder / filename
    fig.savefig(out_path, format="pdf", bbox_inches="tight")
    print(f"[Plot salvo em PDF] -> {out_path.relative_to(REPO_ROOT)}")

print(f"Pandas: {pd.__version__} | NumPy: {np.__version__} | Statsmodels disponível: {HAS_STATSMODELS}")
print(f"Diretório principal de EDA: {EDA_RESULTS_DIR}")
print(f"Diretório exclusivo para cada série temporal: {EDA_SERIES_DIR}")

## 2. Carregamento Otimizado e Dicionário Físico dos Sensores

Todas as 12 colunas de sensores pertencem ao sistema **PH (CBM) 1st Stage** (*Performance Health / Condition-Based Monitoring — 1º Estágio*), organizadas em **4 famílias termodinâmicas**, cada uma com três séries:

| Família Física | Série *Actual* (Medida/Calculada Real) | Série *Expected* (Modelo/Curva de Projeto) | Série *Deviation* (Desvio Relativo %) | Interpretação em Engenharia de Compressores |
| :--- | :--- | :--- | :--- | :--- |
| **Razão de Compressão ($r_p$)** | `ActPress Ratio` | `ExpPress Ratio` | `Press Rat Dev` | Razão $P_{\text{descarga}} / P_{\text{sucção}}$ do 1º estágio. |
| **Head Politrópico ($H_p$)** | `ActCompr Poly Head` | `ExpCompr Poly Head` | `Poly Head Dev` | Energia específica útil transferida ao gás (kJ/kg). |
| **Potência de Eixo ($W_s$)** | `ActShaft Power` | `ExpShaft Power` | `Shft Pwr Dev` | Potência mecânica consumida no eixo (kW). |
| **Eficiência Politrópica ($\eta_p$)** | `ActCompr Poly Eff` | `ExpCompr Poly Eff` | `Poly Eff Dev` | Eficiência termodinâmica do estágio (%). Quedas graduais indicam *fouling* ou desgaste de selos/labirintos. |

In [ ]:
# Resolução segura do caminho do dataset (compatível com execução a partir da raiz ou de eda/)
CANDIDATE_PATHS = [
    Path("/home/jalcantara/DeepLearning2026-3/data/compressor.csv").resolve(),
    (Path.cwd() / "data" / "compressor.csv").resolve(),
    (Path.cwd().parent / "data" / "compressor.csv").resolve(),
]

DATA_PATH = next((p for p in CANDIDATE_PATHS if p.is_file()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Arquivo compressor.csv não encontrado nos caminhos esperados.")

print(f"Carregando arquivo: {DATA_PATH} ({DATA_PATH.stat().st_size / (1024**2):.2f} MB)")

df_raw = pd.read_csv(DATA_PATH, index_col=0)
df_raw.index = pd.to_datetime(df_raw.index, format="ISO8601")
df_raw.index.name = "timestamp"
df_raw = df_raw.sort_index()

# Mapeamento para nomes curtos padronizados (preservando referência aos nomes originais)
COL_MAP = {
    "PH (CBM) 1st Stage ExpPress Ratio": "exp_press_ratio",
    "PH (CBM) 1st Stage Poly Head Dev": "poly_head_dev",
    "PH (CBM) 1st Stage ActShaft Power": "act_shaft_power",
    "PH (CBM) 1st Stage Press Rat Dev": "press_rat_dev",
    "PH (CBM) 1st Stg ActCompr Poly Head": "act_poly_head",
    "PH (CBM) 1st Stage Poly Eff Dev": "poly_eff_dev_raw",
    "PH (CBM) 1st Stage Shft Pwr Dev": "shaft_pwr_dev",
    "PH (CBM) 1st Stage ActCompr Poly Eff": "act_poly_eff",
    "PH (CBM) 1st Stage ExpCompr Poly Eff": "exp_poly_eff",
    "PH (CBM) 1st Stage ExpShaft Power": "exp_shaft_power",
    "PH (CBM) 1st Stage ActPress Ratio": "act_press_ratio",
    "PH (CBM) 1st Stg ExpCompr Poly Head": "exp_poly_head",
}
INV_COL_MAP = {v: k for k, v in COL_MAP.items()}

LABELS_PT = {
    "act_press_ratio": "Razão Pressão Real (-)",
    "exp_press_ratio": "Razão Pressão Esperada (-)",
    "press_rat_dev": "Desvio Razão Pressão (%)",
    "act_poly_head": "Head Politrópico Real (kJ/kg)",
    "exp_poly_head": "Head Politrópico Esperado (kJ/kg)",
    "poly_head_dev": "Desvio Head Politrópico (%)",
    "act_shaft_power": "Potência de Eixo Real (kW)",
    "exp_shaft_power": "Potência de Eixo Esperada (kW)",
    "shaft_pwr_dev": "Desvio Potência de Eixo (%)",
    "act_poly_eff": "Eficiência Politrópica Real (%)",
    "exp_poly_eff": "Eficiência Politrópica Esperada (%)",
    "poly_eff_dev_raw": "Desvio Eficiência (Bruto) (%)",
    "poly_eff_dev": "Desvio Eficiência Reconstruído (%)",
}

df = df_raw.rename(columns=COL_MAP)

print(f"Dimensões brutas : {df.shape[0]:,} linhas x {df.shape[1]} colunas")
print(f"Período coberto  : {df.index.min()}  -->  {df.index.max()}  ({(df.index.max() - df.index.min()).days} dias)")
print(f"Timestamps únicos: {df.index.nunique():,} (duplicatas exatas: {df.index.duplicated().sum()})")
print(f"Uso de memória   : {df.memory_usage(deep=True).sum() / (1024**2):.2f} MB")

display(df.head(5))

## 3. Auditoria de Qualidade de Dados e Diagnóstico do Historiador Industrial

Dados reais de historiadores industriais (como OSIsoft PI / AVEVA) frequentemente apresentam três fenômenos clássicos que distorcem modelos de *Deep Learning* se não forem diagnosticados antes da modelagem:

1. **Tags não configuradas ou mortas** (colunas 100% `NaN`);
2. **Assincronia de varredura (*Split-Row Clock Skew*)**: grupos diferentes de tags calculadas são gravados com 1 a 2 segundos de diferença, dividindo uma única amostra de `30s` em duas linhas parciais (`28s + 2s`);
3. **Compressão por *Deadband* (*Exception/Compression*)**: quando o compressor está parado (valores cravados em `0.0`), o historiador grava pontos apenas a cada `600s` (10 minutos) em vez de `30s`.

In [ ]:
# 3.1 Resumo completo de qualidade e estatísticas brutas por coluna
quality_summary = pd.DataFrame({
    "Tag Original": [INV_COL_MAP[c] for c in df.columns],
    "Válidos": df.notna().sum(),
    "Ausentes (NaN)": df.isna().sum(),
    "% Ausentes": (df.isna().mean() * 100).round(2),
    "Zeros Exatos": (df == 0).sum(),
    "Negativos": (df < 0).sum(),
    "Mínimo": df.min().round(3),
    "Mediana": df.median().round(3),
    "Média": df.mean().round(3),
    "Desvio Padrão": df.std().round(3),
    "Máximo": df.max().round(3),
})
display(quality_summary)

In [ ]:
# 3.2 Diagnóstico da taxa de amostragem (Delta t) e do fenômeno Split-Row (28s + 2s = 30s)
dt_seconds = df.index.to_series().diff().dt.total_seconds()

print("Top 10 intervalos de amostragem (Delta t em segundos):")
dt_dist = dt_seconds.value_counts().head(10).to_frame(name="Ocorrências")
dt_dist["% do Total"] = (dt_dist["Ocorrências"] / len(df) * 100).round(2)
display(dt_dist)

# Inspecionar um exemplo concreto onde dt == 2s (duas linhas consecutivas que se completam)
active_cols_11 = [c for c in df.columns if c != "poly_eff_dev_raw"]
idx_2s = np.where(dt_seconds.values == 2.0)[0]

if len(idx_2s) > 0:
    i_sample = idx_2s[0]
    print(f"Exemplo de Split-Row Assíncrono (índices {i_sample-1} e {i_sample}, separados por 2 segundos):")
    display(df[active_cols_11].iloc[i_sample - 1 : i_sample + 2])

# Plot 1: Distribuição dos Intervalos Delta t (PDF individual)
top_dts = dt_seconds.value_counts().head(6)
fig1, ax1 = plt.subplots(figsize=(8.5, 4.5))
sns.barplot(
    x=[f"{int(k)}s" for k in top_dts.index],
    y=top_dts.values,
    ax=ax1,
    palette="Blues_r",
)
ax1.set_yscale("log")
ax1.set_title(r"Distribuição dos Principais Intervalos $\Delta t$ (Escala Log)")
ax1.set_xlabel("Intervalo entre registros consecutivos")
ax1.set_ylabel("Contagem de registros (log)")
for p in ax1.patches:
    h = p.get_height()
    if h > 0:
        ax1.annotate(f"{int(h):,}", (p.get_x() + p.get_width() / 2, h),
                     ha="center", va="bottom", fontsize=8)
plt.tight_layout()
save_eda_fig(fig1, "01_diagnostico_amostragem_delta_t.pdf")
plt.show()

# Plot 2: Percentual de Valores Ausentes por Variável (PDF individual)
fig2, ax2 = plt.subplots(figsize=(8.5, 4.8))
miss_pct = (df.isna().mean() * 100).sort_values(ascending=True)
miss_pct.plot(kind="barh", ax=ax2, color="#d95f02")
ax2.set_title("Percentual de Valores Ausentes por Variável (Dados Brutos)")
ax2.set_xlabel("% de Valores Ausentes")
ax2.axvline(5.24, color="navy", linestyle="--", linewidth=1.2, label="~5.24% (Split-Row 2s)")
ax2.legend()
plt.tight_layout()
save_eda_fig(fig2, "02_percentual_valores_ausentes.pdf")
plt.show()

### 3.3 Regularização na Grade de `30s` e Reconstrução Analítica de `Poly Eff Dev`

1. **Fusão na grade regular de `30s`**: Ao reamostrar por `resample('30s').mean()`, os pares `(:14, :16)` ou `(:44, :46)` separados por `2s` são fundidos na mesma janela de 30 segundos, eliminando quase integralmente os `NaN`s artificiais das 11 variáveis ativas (**99,82% das janelas populadas ficam 100% completas!**).
2. **Validação da equação dos desvios CBM**: Verificamos se o historiador calcula os desvios percentuais segundo a relação padrão:
   $$\text{Dev (\%)} = \frac{\text{Actual} - \text{Expected}}{\text{Expected}} \times 100$$
   Comprovada a identidade exata nos 3 pares disponíveis (`press_rat_dev`, `poly_head_dev`, `shaft_pwr_dev`), **reconstruímos fisicamente** a coluna ausente `poly_eff_dev`!

In [ ]:
# Reamostragem para grade estritamente regular de 30 segundos
df_30s = df[active_cols_11].resample("30s").mean()

populated_buckets = df_30s.notna().any(axis=1).sum()
complete_buckets = df_30s.notna().all(axis=1).sum()

print(f"Total de janelas regulares de 30s em 182 dias : {len(df_30s):,}")
print(f"Janelas com pelo menos 1 leitura              : {populated_buckets:,}")
print(f"Janelas completas nas 11 variáveis ativas     : {complete_buckets:,} ({complete_buckets / populated_buckets * 100:.2f}% das populadas!)")
print(f"Linhas extras eliminadas pela fusão de 2s     : {len(df) - populated_buckets:,}")

# Validação matemática da fórmula Dev = (Actual - Expected) / Expected * 100
valid_check = df_30s.dropna().query("act_shaft_power > 5000 and 50 <= act_poly_eff <= 100")

formula_checks = []
pairs = [
    ("Razão de Compressão", "act_press_ratio", "exp_press_ratio", "press_rat_dev"),
    ("Head Politrópico", "act_poly_head", "exp_poly_head", "poly_head_dev"),
    ("Potência de Eixo", "act_shaft_power", "exp_shaft_power", "shaft_pwr_dev"),
]

for nome, act_col, exp_col, dev_col in pairs:
    calc_dev = (valid_check[act_col] - valid_check[exp_col]) / valid_check[exp_col] * 100.0
    err = valid_check[dev_col] - calc_dev
    formula_checks.append({
        "Família Física": nome,
        "Viés Médio (p.p.)": round(err.mean(), 5),
        "MAE (p.p.)": round(err.abs().mean(), 5),
        "RMSE (p.p.)": round(np.sqrt((err ** 2).mean()), 5),
        "R² entre Real e Fórmula": round(np.corrcoef(valid_check[dev_col], calc_dev)[0, 1] ** 2, 6),
    })

print("\nValidação da Equação Física dos Desvios CBM:")
display(pd.DataFrame(formula_checks))

# Reconstrução segura de poly_eff_dev (evitando divisão por zero em paradas)
exp_eff_safe = df_30s["exp_poly_eff"].where(df_30s["exp_poly_eff"] > 1.0, np.nan)
df_30s["poly_eff_dev"] = (df_30s["act_poly_eff"] - exp_eff_safe) / exp_eff_safe * 100.0
df_30s.loc[df_30s["exp_poly_eff"] == 0.0, "poly_eff_dev"] = 0.0

SENSOR_COLS = [
    "act_press_ratio", "exp_press_ratio", "press_rat_dev",
    "act_poly_head", "exp_poly_head", "poly_head_dev",
    "act_shaft_power", "exp_shaft_power", "shaft_pwr_dev",
    "act_poly_eff", "exp_poly_eff", "poly_eff_dev",
]
df_30s = df_30s[SENSOR_COLS]
print("Coluna 'poly_eff_dev' reconstruída com sucesso! Total de variáveis analíticas:", len(SENSOR_COLS))

### 3.4 Identificação de Paradas, Lacunas de Compressão e *Glitches* Físicos de Partida

Classificamos cada instante da grade de `30s` em quatro estados mutuamente exclusivos:
- **`Gap / Sem Leitura`**: janelas sem leitura (incluindo compressão de 10 min durante paradas ou manutenção do servidor PI);
- **`Parada (Offline)`**: potência ou razão de compressão nula/próxima de zero (`act_shaft_power <= 500 kW` ou `act_press_ratio < 1.05`);
- **`Transiente / Glitch`**: leituras durante rampas de partida/parada onde a divisão por potência próxima de zero gera valores não-físicos (`act_poly_eff < 50%`, `act_poly_eff > 100%` ou `act_shaft_power < 4000 kW`);
- **`Operação Normal`**: regime produtivo contínuo do compressor.

In [ ]:
# Classificação de estado operacional na grade de 30s
state = pd.Series("Operação Normal", index=df_30s.index, name="estado_operacional")

is_missing = df_30s[["act_shaft_power", "act_press_ratio", "act_poly_eff"]].isna().any(axis=1)
is_offline = (~is_missing) & ((df_30s["act_shaft_power"] <= 500.0) | (df_30s["act_press_ratio"] < 1.05))
is_glitch = (~is_missing) & (~is_offline) & (
    (df_30s["act_poly_eff"] < 50.0)
    | (df_30s["act_poly_eff"] > 100.0)
    | (df_30s["act_shaft_power"] < 4000.0)
)

state.loc[is_glitch] = "Transiente / Glitch"
state.loc[is_offline] = "Parada (Offline)"
state.loc[is_missing] = "Gap / Sem Leitura"

state_counts = state.value_counts().to_frame("Janelas (30s)")
state_counts["Horas Equivalentes"] = (state_counts["Janelas (30s)"] * 30 / 3600).round(2)
state_counts["% do Tempo"] = (state_counts["Janelas (30s)"] / len(state) * 100).round(3)
print("Resumo dos Estados Operacionais na Grade Regular (30s):")
display(state_counts)

# Detectar blocos contínuos de não-operação (Paradas + Gaps)
not_normal = state != "Operação Normal"
block_id = (not_normal != not_normal.shift()).cumsum()
blocks = (
    df_30s[not_normal]
    .assign(estado=state[not_normal], block=block_id[not_normal])
    .groupby("block")
    .agg(
        inicio=("act_shaft_power", lambda s: s.index.min()),
        fim=("act_shaft_power", lambda s: s.index.max()),
        janelas_30s=("act_shaft_power", "size"),
        min_power=("act_shaft_power", "min"),
        estados=("estado", lambda s: ", ".join(s.unique())),
    )
)
blocks["duracao_min"] = ((blocks["fim"] - blocks["inicio"]).dt.total_seconds() / 60.0 + 0.5).round(1)
major_events = blocks.sort_values("duracao_min", ascending=False).head(10).reset_index(drop=True)
print("\nTop 10 Maiores Eventos de Parada / Lacuna no Período de 6 Meses:")
display(major_events)

# Exibir os registros mais extremos de glitch físico (ex.: rampa de partida pós-parada)
print("\nExemplos de Glitches Termodinâmicos durante Rampas de Partida/Parada:")
glitches_df = df_30s.loc[state == "Transiente / Glitch", [
    "act_shaft_power", "act_press_ratio", "act_poly_head", "act_poly_eff", "poly_eff_dev"
]].sort_values("act_poly_eff", ascending=False)
display(pd.concat([glitches_df.head(5), glitches_df.tail(5)]))

## 4. Visualização Multivariada das Séries Temporais

Para evitar poluição visual e artefatos de escala provocados pelos *glitches* de partida (onde `act_poly_eff` atinge `5.879%` e `-742%`), criamos:
- `df_clean_30s`: grade de `30s` filtrada apenas em **`Operação Normal`** (com interpolação curta de até 2 minutos para pequenas falhas isoladas de amostragem);
- `df_1h` e `df_1d`: agregações horária e diária para análise clara da tendência de longo prazo ao longo dos 182 dias;
- Um painel de **zoom de alta resolução (`30s`)** focado especificamente na grande parada de `24–25/Set/2019` para inspecionar a física da parada e rampa de partida.

In [ ]:
# Construção das visões limpa (30s), 5 minutos, horária (1h) e diária (1D)
df_norm_only = df_30s.where(state == "Operação Normal")
df_clean_30s = df_norm_only.interpolate(method="time", limit=4)  # interpola apenas micro-gaps <= 2 min

df_5min = df_norm_only.resample("5min").mean()
df_1h = df_norm_only.resample("1h").mean()
df_1d = df_norm_only.resample("1D").mean()

shutdown_dates = [
    pd.Timestamp("2019-09-12 18:40"),
    pd.Timestamp("2019-09-24 05:10"),
    pd.Timestamp("2019-10-10 10:40"),
]

# 4.1 Plot de CADA UMA das 12 Séries Temporais individualmente salvo na pasta 'series_temporais/'
sensor_colors = {
    "act_press_ratio": "#1f77b4", "exp_press_ratio": "#ff7f0e", "press_rat_dev": "#1b9e77",
    "act_poly_head": "#2ca02c", "exp_poly_head": "#d62728", "poly_head_dev": "#d95f02",
    "act_shaft_power": "#9467bd", "exp_shaft_power": "#8c564b", "shaft_pwr_dev": "#7570b3",
    "act_poly_eff": "#008080", "exp_poly_eff": "#e377c2", "poly_eff_dev": "#e7298a",
}

for col in SENSOR_COLS:
    fig, ax = plt.subplots(figsize=(12, 4.2))
    cor = sensor_colors.get(col, "#2b8cbe")
    
    # Plot em escala horária e média diária para leitura limpa e detalhada
    ax.plot(df_1h.index, df_1h[col], color=cor, alpha=0.35, linewidth=0.8, label="Média Horária (1h)")
    ax.plot(df_1d.index, df_1d[col], color=cor, linewidth=2.0, label="Tendência Diária (1D)")

    for i_sd, sd_ts in enumerate(shutdown_dates):
        ax.axvline(sd_ts, color="crimson", linestyle=":", alpha=0.75,
                   label="Parada Operacional" if i_sd == 0 else None)

    ax.set_title(f"Série Temporal — {LABELS_PT[col]} (Set/2019 a Mar/2020)")
    ax.set_ylabel(LABELS_PT[col])
    ax.set_xlabel("Data")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
    ax.legend(loc="upper right")
    plt.tight_layout()
    save_eda_fig(fig, f"serie_{col}.pdf", subfolder=EDA_SERIES_DIR)
    plt.show()

print("Todas as 12 séries temporais individuais salvas em PDF em results/eda/series_temporais/!")

In [ ]:
# 4.2 Evolução dos 4 Desvios Percentuais CBM (Média Móvel de 24h e Tendência de 7d em figuras separadas)
dev_cols = ["press_rat_dev", "poly_head_dev", "shaft_pwr_dev", "poly_eff_dev"]
dev_rolling_24h = df_1h[dev_cols].rolling(window=24, min_periods=6).mean()
dev_rolling_7d = df_1h[dev_cols].rolling(window=24 * 7, min_periods=24).mean()

palette_dev = ["#1b9e77", "#d95f02", "#7570b3", "#e7298a"]

# Plot 1: Média Móvel de 24 Horas
fig1, ax1 = plt.subplots(figsize=(12, 4.5))
for col, cor in zip(dev_cols, palette_dev):
    ax1.plot(dev_rolling_24h.index, dev_rolling_24h[col], label=LABELS_PT[col], color=cor, linewidth=1.3, alpha=0.85)
ax1.axhline(0, color="black", linestyle="--", linewidth=0.9)
ax1.set_title("Evolução dos 4 Desvios CBM — Média Móvel de 24 Horas")
ax1.set_ylabel("Desvio Relativo (%)")
ax1.set_xlabel("Data")
ax1.legend(loc="upper right", ncol=2)
plt.tight_layout()
save_eda_fig(fig1, "04_desvios_cbm_media_movel_24h.pdf")
plt.show()

# Plot 2: Tendência Estrutural de 7 Dias com Ciclos de Degradação
fig2, ax2 = plt.subplots(figsize=(12, 4.5))
for col, cor in zip(dev_cols, palette_dev):
    ax2.plot(dev_rolling_7d.index, dev_rolling_7d[col], label=LABELS_PT[col], color=cor, linewidth=2.2)
ax2.axhline(0, color="black", linestyle="--", linewidth=0.9)
ax2.axvspan(pd.Timestamp("2019-09-01"), pd.Timestamp("2019-11-10"), color="#fee0d2", alpha=0.28, label="Ciclo 1: Queda gradual de desempenho")
ax2.axvspan(pd.Timestamp("2019-11-20"), pd.Timestamp("2019-12-20"), color="#e5f5e0", alpha=0.35, label="Recuperação / Mudança de Ponto Operacional")
ax2.axvspan(pd.Timestamp("2019-12-20"), pd.Timestamp("2020-03-01"), color="#deebf7", alpha=0.30, label="Ciclo 2: Segundo período de deriva gradual")
ax2.set_title("Tendência Estrutural dos Desvios CBM — Média Móvel de 7 Dias")
ax2.set_ylabel("Desvio Relativo (%)")
ax2.set_xlabel("Data")
ax2.legend(loc="upper right", ncol=2)
plt.tight_layout()
save_eda_fig(fig2, "05_desvios_cbm_tendencia_7d.pdf")
plt.show()

In [ ]:
# 4.3 Zoom de Alta Resolução (30s): Parada de 24-25/Set/2019 (3 figuras individuais)
zoom_start, zoom_end = "2019-09-24 03:00:00", "2019-09-25 10:00:00"
df_zoom = df_30s.loc[zoom_start:zoom_end]

# Plot 1: Potência Real e Esperada (mostrando pontos espaçados em 10 min por Deadband)
fig1, ax1 = plt.subplots(figsize=(11, 4.0))
ax1.plot(df_zoom.index, df_zoom["act_shaft_power"], marker=".", markersize=2.5, linewidth=1.0, color="#9467bd", label="Potência Real (kW)")
ax1.plot(df_zoom.index, df_zoom["exp_shaft_power"], linewidth=1.0, linestyle="--", color="#8c564b", label="Potência Esperada (kW)")
ax1.set_title("Zoom (30s) na Parada de 24–25/Set/2019 — Potência de Eixo (Compressão Deadband)")
ax1.set_ylabel("Potência (kW)")
ax1.set_xlabel("Timestamp")
ax1.legend(loc="upper right")
plt.tight_layout()
save_eda_fig(fig1, "06_zoom_parada_potencia_eixo.pdf")
plt.show()

# Plot 2: Razão de Compressão Real
fig2, ax2 = plt.subplots(figsize=(11, 3.8))
ax2.plot(df_zoom.index, df_zoom["act_press_ratio"], marker=".", markersize=2.5, linewidth=1.0, color="#1f77b4", label="Razão Pressão Real")
ax2.set_title("Zoom (30s) na Parada de 24–25/Set/2019 — Razão de Pressão Real")
ax2.set_ylabel("Razão de Pressão")
ax2.set_xlabel("Timestamp")
ax2.legend(loc="upper right")
plt.tight_layout()
save_eda_fig(fig2, "07_zoom_parada_razao_pressao.pdf")
plt.show()

# Plot 3: Eficiência Bruta e Transiente com Glitch de Partida
fig3, ax3 = plt.subplots(figsize=(11, 4.2))
ax3.plot(df_zoom.index, df_zoom["act_poly_eff"], marker="o", markersize=3, linewidth=1.0, color="crimson", label="Eficiência Bruta (%) — Sem Filtro")
ax3.set_ylim(-120, 260)
ax3.axhspan(70, 95, color="seagreen", alpha=0.15, label="Faixa Física Normal (70%–95%)")
ax3.set_title("Transiente de Partida (25/Set 07:09–07:15): Spike Artificial por Potência Próxima de Zero")
ax3.set_ylabel("Eficiência (%) [truncada]")
ax3.set_xlabel("Timestamp")
ax3.legend(loc="upper left")
plt.tight_layout()
save_eda_fig(fig3, "08_zoom_parada_glitch_eficiencia.pdf")
plt.show()

## 5. Distribuições Estatísticas, Sazonalidade Multi-Escala e Estacionariedade

Nesta seção investigamos:
1. **Assimetria (*Skewness*), Curtose (*Kurtosis*) e quantis** em regime de operação normal;
2. **Sazonalidade diurna (Hora do Dia) e mensal (Mês × Hora)**: compressores a gás sofrem influência térmica da temperatura ambiente de sucção/intercoolers entre madrugada e tarde;
3. **Decomposição STL (Seasonal-Trend decomposition using LOESS)** e testes estatísticos formais **ADF** ($H_0$: raiz unitária / não-estacionária) e **KPSS** ($H_0$: estacionária em torno de nível/tendência).

In [ ]:
# 5.1 Estatísticas descritivas detalhadas em Operação Normal (sem paradas nem glitches)
df_norm_valid = df_norm_only.dropna()

desc_stats = pd.DataFrame({
    "Média": df_norm_valid.mean(),
    "Desvio Padrão": df_norm_valid.std(),
    "CV (%)": (df_norm_valid.std() / df_norm_valid.mean().abs() * 100),
    "P01": df_norm_valid.quantile(0.01),
    "P25": df_norm_valid.quantile(0.25),
    "Mediana (P50)": df_norm_valid.quantile(0.50),
    "P75": df_norm_valid.quantile(0.75),
    "P99": df_norm_valid.quantile(0.99),
    "IQR": df_norm_valid.quantile(0.75) - df_norm_valid.quantile(0.25),
    "Assimetria (Skew)": df_norm_valid.skew(),
    "Curtose (Kurt)": df_norm_valid.kurt(),
}).round(3)

print(f"Estatísticas em Operação Normal (N = {len(df_norm_valid):,} janelas de 30s):")
display(desc_stats)

sample_dist = df_norm_valid.sample(n=min(40_000, len(df_norm_valid)), random_state=RANDOM_SEED)

# Salvar distribuição individual em PDF para cada variável principal
for col in SENSOR_COLS:
    fig, ax = plt.subplots(figsize=(6.5, 4.0))
    sns.histplot(sample_dist[col], bins=45, kde=True, ax=ax, color="#2b8cbe", edgecolor="none", stat="density")
    ax.axvline(sample_dist[col].mean(), color="crimson", linestyle="--", linewidth=1.2, label="Média")
    ax.axvline(sample_dist[col].median(), color="black", linestyle=":", linewidth=1.2, label="Mediana")
    ax.set_title(f"Distribuição — {LABELS_PT[col]}", fontsize=10.5)
    ax.set_xlabel(LABELS_PT[col])
    ax.set_ylabel("Densidade")
    ax.legend(fontsize=8.5)
    plt.tight_layout()
    save_eda_fig(fig, f"09_distribuicao_{col}.pdf")
    plt.close(fig)

print("Distribuições univariadas salvas individualmente em PDF para todas as 12 variáveis!")

# Boxplot das séries padronizadas (Z-score) para comparar caudas e outliers relativos
z_sample = pd.DataFrame(
    StandardScaler().fit_transform(sample_dist),
    columns=[LABELS_PT[c] for c in SENSOR_COLS],
)
fig_box, ax_box = plt.subplots(figsize=(12, 5.5))
sns.boxplot(data=z_sample, orient="h", fliersize=1.5, palette="vlag", ax=ax_box)
ax_box.axvline(0, color="black", linestyle="--", linewidth=0.9)
ax_box.set_xlim(-5.5, 5.5)
ax_box.set_title("Comparação das Distribuições Padronizadas (Z-Score) e Caudas em Operação Normal")
ax_box.set_xlabel("Z-Score (Desvios Padrão em relação à Média)")
plt.tight_layout()
save_eda_fig(fig_box, "10_boxplots_zscore_padronizados.pdf")
plt.show()

In [ ]:
# 5.2 Sazonalidade Multi-Escala: Figuras individuais para cada padrão temporal
df_seas = df_1h.dropna().copy()
df_seas["hora"] = df_seas.index.hour
df_seas["dia_semana"] = df_seas.index.dayofweek
df_seas["ano_mes"] = df_seas.index.strftime("%Y-%m")

dias_pt = ["Seg", "Ter", "Qua", "Qui", "Sex", "Sáb", "Dom"]

# Plot 1: Ciclo Diurno Médio (Hora do Dia com IC 95%)
fig1, ax1 = plt.subplots(figsize=(9, 4.5))
hourly_grp = df_seas.groupby("hora")[["act_shaft_power", "act_poly_eff"]].agg(["mean", "sem"])
h_idx = hourly_grp.index
p_mean, p_sem = hourly_grp[("act_shaft_power", "mean")], hourly_grp[("act_shaft_power", "sem")]
e_mean, e_sem = hourly_grp[("act_poly_eff", "mean")], hourly_grp[("act_poly_eff", "sem")]

ax1_r = ax1.twinx()
ax1.plot(h_idx, p_mean, color="#9467bd", marker="o", linewidth=2, label="Potência Real (kW)")
ax1.fill_between(h_idx, p_mean - 1.96 * p_sem, p_mean + 1.96 * p_sem, color="#9467bd", alpha=0.18)
ax1_r.plot(h_idx, e_mean, color="#008080", marker="s", linewidth=2, linestyle="--", label="Eficiência Real (%)")
ax1_r.fill_between(h_idx, e_mean - 1.96 * e_sem, e_mean + 1.96 * e_sem, color="#008080", alpha=0.15)
ax1.set_title("Ciclo Diurno Médio (IC 95%): Potência vs. Eficiência por Hora")
ax1.set_xlabel("Hora do Timestamp (0h–23h)")
ax1.set_ylabel("Potência de Eixo (kW)", color="#9467bd")
ax1_r.set_ylabel("Eficiência Politrópica (%)", color="#008080")
ax1_r.grid(False)
plt.tight_layout()
save_eda_fig(fig1, "11_sazonalidade_ciclo_diurno.pdf")
plt.show()

# Plot 2: Perfil por Dia da Semana
fig2, ax2 = plt.subplots(figsize=(8.5, 4.5))
sns.boxplot(data=df_seas, x="dia_semana", y="act_poly_eff", ax=ax2, palette="crest", showfliers=False)
ax2.set_xticks(range(7))
ax2.set_xticklabels(dias_pt)
ax2.set_title("Distribuição da Eficiência Politrópica por Dia da Semana")
ax2.set_xlabel("Dia da Semana")
ax2.set_ylabel("Eficiência Politrópica Real (%)")
plt.tight_layout()
save_eda_fig(fig2, "12_sazonalidade_dia_da_semana.pdf")
plt.show()

# Plot 3: Evolução Mensal (Boxplots)
fig3, ax3 = plt.subplots(figsize=(8.5, 4.5))
sns.boxplot(data=df_seas, x="ano_mes", y="act_poly_eff", ax=ax3, palette="Spectral", showfliers=False)
ax3.set_title("Evolução Mensal da Eficiência Politrópica Real")
ax3.set_xlabel("Ano-Mês")
ax3.set_ylabel("Eficiência Politrópica Real (%)")
plt.tight_layout()
save_eda_fig(fig3, "13_sazonalidade_evolucao_mensal.pdf")
plt.show()

# Plot 4: Heatmap Mês x Hora
fig4, ax4 = plt.subplots(figsize=(10, 4.5))
piv_mh = df_seas.pivot_table(index="ano_mes", columns="hora", values="act_poly_eff", aggfunc="mean")
sns.heatmap(piv_mh, cmap="YlGnBu", ax=ax4, cbar_kws={"label": "Eficiência Média (%)"})
ax4.set_title("Mapa de Calor Sazonal (Ano-Mês × Hora) — Eficiência Politrópica")
ax4.set_xlabel("Hora do Dia")
ax4.set_ylabel("Ano-Mês")
plt.tight_layout()
save_eda_fig(fig4, "14_sazonalidade_heatmap_mes_hora.pdf")
plt.show()

In [ ]:
# 5.3 Decomposição STL (Componentes individuais salvos em PDF) e Testes de Estacionariedade
eff_1h_interp = df_1h["act_poly_eff"].interpolate(method="time").bfill().ffill()

if HAS_STATSMODELS:
    stl_res = STL(eff_1h_interp, period=24, robust=True).fit()
    trend_comp = stl_res.trend
    seas_comp = stl_res.seasonal
    resid_comp = stl_res.resid
else:
    trend_comp = eff_1h_interp.rolling(window=24 * 7, center=True, min_periods=24).mean().bfill().ffill()
    detrended = eff_1h_interp - trend_comp
    hourly_map = detrended.groupby(detrended.index.hour).transform("mean")
    seas_comp = hourly_map
    resid_comp = detrended - seas_comp

stl_plots = [
    ("15_stl_serie_original.pdf", "Série Horária Observada — Eficiência Politrópica Real (%)", eff_1h_interp, "#008080"),
    ("16_stl_tendencia.pdf", "Componente de Tendência — Degradação e Deriva Estrutural", trend_comp, "#d95f02"),
    ("17_stl_sazonalidade_diurna.pdf", "Componente Sazonal Diurna (Período = 24h)", seas_comp, "#1f77b4"),
    ("18_stl_residuos.pdf", "Componente Residual — Ruído de Alta Frequência", resid_comp, "#555555"),
]

for filename, title, series_data, cor in stl_plots:
    fig, ax = plt.subplots(figsize=(11, 3.8))
    ax.plot(series_data.index, series_data, color=cor, linewidth=1.1)
    if "Residual" in title:
        ax.axhline(0, color="crimson", linestyle="--", linewidth=0.9)
    ax.set_title(title)
    ax.set_xlabel("Data")
    plt.tight_layout()
    save_eda_fig(fig, filename)
    plt.show()

# Testes de estacionariedade ADF e KPSS na escala horária (4.368 observações)
if HAS_STATSMODELS:
    stat_rows = []
    for col in SENSOR_COLS:
        s_1h = df_1h[col].interpolate(method="time").dropna()
        adf_stat, adf_p, _, _, _, _ = adfuller(s_1h, maxlag=48, autolag="AIC")
        kpss_stat, kpss_p, _, _ = kpss(s_1h, regression="c", nlags="auto")

        if adf_p < 0.05 and kpss_p >= 0.05:
            diag = "Estacionária"
        elif adf_p >= 0.05 and kpss_p < 0.05:
            diag = "Não-Estacionária (Raiz Unitária / Deriva)"
        elif adf_p < 0.05 and kpss_p < 0.05:
            diag = "Não-Estacionária em Nível (Mudança de Regime / Tendência Local)"
        else:
            diag = "Inconclusivo"

        stat_rows.append({
            "Variável": LABELS_PT[col],
            "ADF Stat": round(adf_stat, 3),
            "ADF p-valor": round(adf_p, 4),
            "KPSS Stat": round(kpss_stat, 3),
            "KPSS p-valor": round(kpss_p, 4),
            "Diagnóstico Conjunto (ADF + KPSS)": diag,
        })

    print("Testes Formais de Estacionariedade (Séries Horárias — N = 4.368h):")
    display(pd.DataFrame(stat_rows))

## 6. Dinâmica Temporal: Memória (ACF/PACF), Defasagem (*Lead-Lag*) e Espectro de Frequências

Para projetar janelas de entrada (*lookback windows*) em redes neurais temporais (LSTM, TCN, Transformer), precisamos responder a três perguntas fundamentais:
1. **Qual é o horizonte de memória autocorrelacionada do processo?** (ACF e PACF em curto prazo de `30s` e médio prazo de `1h`);
2. **Existe atraso dinâmico (*lead-lag*) entre potência/pressão e eficiência?** (Correlação Cruzada);
3. **Quais frequências cíclicas dominam a variância das séries?** (Densidade Espectral de Potência de Welch).

In [ ]:
# 6.1 Funções de Autocorrelação (ACF) e Autocorrelação Parcial (PACF)
def compute_acf_fallback(x: np.ndarray, nlags: int) -> np.ndarray:
    x_c = x - np.mean(x)
    denom = np.dot(x_c, x_c)
    return np.array([1.0] + [np.dot(x_c[:-k], x_c[k:]) / denom for k in range(1, nlags + 1)])

nlags_1h = 168  # 7 dias
lags_h = np.arange(nlags_1h + 1)

# Plot de Autocorrelação (ACF) individual para CADA uma das 12 variáveis
for col in SENSOR_COLS:
    s_col_1h = df_1h[col].interpolate(method="time").dropna().values
    if HAS_STATSMODELS:
        acf_vals = acf(s_col_1h, nlags=nlags_1h, fft=True)
    else:
        acf_vals = compute_acf_fallback(s_col_1h, nlags_1h)

    fig, ax = plt.subplots(figsize=(8.5, 4.2))
    ax.plot(lags_h, acf_vals, color="#1f77b4", linewidth=1.8, label="ACF")
    conf = 1.96 / np.sqrt(len(s_col_1h))
    ax.axhspan(-conf, conf, color="orange", alpha=0.2, label="IC 95%")
    for d in range(1, 8):
        ax.axvline(d * 24, color="crimson", linestyle=":", alpha=0.45)
    ax.set_title(f"Autocorrelação (ACF até 7 dias) — {LABELS_PT[col]}")
    ax.set_xlabel("Defasagem (horas)")
    ax.set_ylabel("Autocorrelação")
    ax.legend(loc="upper right")
    plt.tight_layout()
    save_eda_fig(fig, f"20_autocorrelacao_acf_{col}.pdf")
    plt.close(fig)

print("Plots individuais de autocorrelação (ACF) salvos em PDF para todas as 12 variáveis!")

# Plot detalhado: ACF de curto prazo (30s) na potência
sub_30s = df_clean_30s.loc["2019-12-01":"2019-12-08", "act_shaft_power"].dropna().values
acf_pwr_30s = acf(sub_30s, nlags=120, fft=True) if HAS_STATSMODELS else compute_acf_fallback(sub_30s, 120)
fig19, ax19 = plt.subplots(figsize=(8.5, 4.2))
lags_min = np.arange(len(acf_pwr_30s)) * 0.5
ax19.plot(lags_min, acf_pwr_30s, color="#9467bd", linewidth=2)
ax19.axhline(1.96 / np.sqrt(len(sub_30s)), color="gray", linestyle="--", linewidth=0.9)
ax19.set_title("ACF de Curto Prazo (Grade 30s: 0 a 60 min) — Potência de Eixo Real")
ax19.set_xlabel("Defasagem (minutos)")
ax19.set_ylabel("Autocorrelação")
plt.tight_layout()
save_eda_fig(fig19, "19_dinamica_acf_curto_prazo_30s.pdf")
plt.show()

# Plot detalhado: PACF horária na eficiência
s_eff_1h = df_1h["act_poly_eff"].interpolate(method="time").dropna().values
pacf_eff_1h = pacf(s_eff_1h, nlags=48, method="ywm") if HAS_STATSMODELS else compute_acf_fallback(np.diff(s_eff_1h), 48)
fig21, ax21 = plt.subplots(figsize=(8.5, 4.2))
ax21.stem(np.arange(len(pacf_eff_1h)), pacf_eff_1h, basefmt=" ")
conf_eff = 1.96 / np.sqrt(len(s_eff_1h))
ax21.axhspan(-conf_eff, conf_eff, color="orange", alpha=0.2, label="IC 95%")
ax21.set_title("PACF Horária (0 a 48h) — Eficiência Real")
ax21.set_xlabel("Defasagem (horas)")
ax21.set_ylabel("Autocorrelação Parcial")
ax21.legend()
plt.tight_layout()
save_eda_fig(fig21, "21_dinamica_pacf_horaria.pdf")
plt.show()

In [ ]:
# 6.2 Correlação Cruzada (Lead-Lag) e Análise Espectral de Welch em figuras individuais
diff_5m = df_5min[["act_shaft_power", "act_press_ratio", "act_poly_eff"]].interpolate(limit=3).diff().dropna()

def cross_corr(x: pd.Series, y: pd.Series, max_lag: int = 36):
    x_z = (x.values - x.mean()) / x.std()
    y_z = (y.values - y.mean()) / y.std()
    lags = np.arange(-max_lag, max_lag + 1)
    cc = []
    n = len(x_z)
    for k in lags:
        if k < 0:
            cc.append(np.dot(x_z[:k], y_z[-k:]) / n)
        elif k > 0:
            cc.append(np.dot(x_z[k:], y_z[:-k]) / n)
        else:
            cc.append(np.dot(x_z, y_z) / n)
    return lags, np.array(cc)

lags_5m, cc_pwr_pr = cross_corr(diff_5m["act_shaft_power"], diff_5m["act_press_ratio"], max_lag=24)
_, cc_pr_eff = cross_corr(diff_5m["act_press_ratio"], diff_5m["act_poly_eff"], max_lag=24)

# Plot 1: Correlação Cruzada (Lead-Lag)
fig1, ax1 = plt.subplots(figsize=(8.5, 4.5))
lags_minutes = lags_5m * 5
ax1.plot(lags_minutes, cc_pwr_pr, marker=".", label=r"$\Delta$ Potência $\leftrightarrow$ $\Delta$ Razão de Pressão", color="#1f77b4")
ax1.plot(lags_minutes, cc_pr_eff, marker=".", label=r"$\Delta$ Razão de Pressão $\leftrightarrow$ $\Delta$ Eficiência", color="#008080")
ax1.axvline(0, color="black", linestyle="--", linewidth=1)
ax1.set_title("Correlação Cruzada (Lead-Lag) das Variações (5 min)")
ax1.set_xlabel("Defasagem (minutos) [Lag > 0: 1ª variável antecede a 2ª]")
ax1.set_ylabel("Correlação Cruzada")
ax1.legend()
plt.tight_layout()
save_eda_fig(fig1, "22_dinamica_correlacao_cruzada_lead_lag.pdf")
plt.show()

# Plot 2: Densidade Espectral de Potência de Welch
s_spec_eff = signal.detrend(df_1h["act_poly_eff"].interpolate(method="time").bfill().ffill().values)
s_spec_pwr = signal.detrend(df_1h["act_shaft_power"].interpolate(method="time").bfill().ffill().values)
freqs_cpd, psd_eff = signal.welch(s_spec_eff, fs=24.0, nperseg=512)
_, psd_pwr = signal.welch(s_spec_pwr, fs=24.0, nperseg=512)

fig2, ax2 = plt.subplots(figsize=(8.5, 4.5))
ax2.semilogy(freqs_cpd[1:], psd_eff[1:] / psd_eff[1:].max(), color="#008080", linewidth=1.8, label="PSD Normalizada — Eficiência")
ax2.semilogy(freqs_cpd[1:], psd_pwr[1:] / psd_pwr[1:].max(), color="#9467bd", linewidth=1.4, alpha=0.8, label="PSD Normalizada — Potência")
ax2.axvline(1.0, color="crimson", linestyle="--", alpha=0.8, label="1 ciclo/dia (Período 24h)")
ax2.axvline(2.0, color="darkorange", linestyle=":", alpha=0.8, label="2 ciclos/dia (Harmônico 12h)")
ax2.set_xlim(0, 6)
ax2.set_title("Densidade Espectral de Potência (Método de Welch)")
ax2.set_xlabel("Frequência (ciclos por dia)")
ax2.set_ylabel("Potência Espectral Relativa (log)")
ax2.legend()
plt.tight_layout()
save_eda_fig(fig2, "23_dinamica_espectro_welch_periodicidades.pdf")
plt.show()

## 7. Análise Multivariada, Mapas Termodinâmicos, PCA (MSPC) e Regimes Operacionais

Como as 12 séries representam grandezas termodinamicamente acopladas de um mesmo estágio de compressão, esperamos **forte multicolinearidade estrutural** (ex.: Head Politrópico esperado vs. Razão de Compressão esperada têm correlação quase unitária pela equação politrópica $H_p = \frac{Z R T_1}{(n-1)/n}[r_p^{(n-1)/n} - 1]$).

Nesta seção avaliamos:
1. **Correlação linear (Pearson) vs. monotônica (Spearman)** e dendrograma de agrupamento de sensores;
2. **Mapas de performance termodinâmica** do compressor;
3. **PCA + Monitoramento Estatístico Multivariado de Processo (MSPC)** via estatísticas **$T^2$ de Hotelling** (variação dentro do subespaço físico normal) e **SPE / Estatística $Q$** (quebra das relações físicas entre sensores!);
4. **Identificação não supervisionada de Regimes Operacionais** via *K-Means*.

In [ ]:
# 7.1 Matrizes de Correlação (Pearson e Spearman) e Dendrograma de Sensores
corr_pearson = df_norm_valid.corr(method="pearson")
corr_spearman = df_norm_valid.sample(n=min(50_000, len(df_norm_valid)), random_state=RANDOM_SEED).corr(method="spearman")

# Plot 1: Matriz de Correlação Linear de Pearson (Heatmap Completo)
fig1, ax1 = plt.subplots(figsize=(10, 8.5))
mask_tri = np.triu(np.ones_like(corr_pearson, dtype=bool), k=1)
sns.heatmap(
    corr_pearson,
    mask=mask_tri,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    vmin=-1,
    vmax=1,
    annot_kws={"size": 8.0},
    ax=ax1,
    cbar_kws={"shrink": 0.8, "label": "Coeficiente de Pearson (r)"},
)
ax1.set_title("Matriz de Correlação Linear de Pearson — 12 Sensores CBM (Operação Normal)", fontsize=11, fontweight="bold")
plt.tight_layout()
save_eda_fig(fig1, "24_matriz_correlacao_pearson.pdf")
plt.show()

# Plot 2: Matriz de Correlação Monotônica de Spearman
fig2, ax2 = plt.subplots(figsize=(10, 8.5))
sns.heatmap(
    corr_spearman,
    mask=mask_tri,
    annot=True,
    fmt=".2f",
    cmap="vlag",
    vmin=-1,
    vmax=1,
    annot_kws={"size": 8.0},
    ax=ax2,
    cbar_kws={"shrink": 0.8, "label": "Coeficiente de Spearman (rho)"},
)
ax2.set_title("Matriz de Correlação Monotônica de Spearman — 12 Sensores CBM", fontsize=11, fontweight="bold")
plt.tight_layout()
save_eda_fig(fig2, "24_matriz_correlacao_spearman.pdf")
plt.show()

# Plot 3: Dendrograma Hierárquico dos Sensores
dist_mat = 1.0 - np.abs(corr_pearson.values)
np.fill_diagonal(dist_mat, 0.0)
dist_mat = np.clip((dist_mat + dist_mat.T) / 2.0, 0.0, None)
linkage_z = hierarchy.linkage(squareform(dist_mat), method="average")

fig3, ax3 = plt.subplots(figsize=(9, 6.0))
hierarchy.dendrogram(
    linkage_z,
    labels=[LABELS_PT[c] for c in SENSOR_COLS],
    orientation="right",
    ax=ax3,
    color_threshold=0.25,
)
ax3.set_title(r"Agrupamento Hierárquico dos Sensores (Distância = $1 - |r|$)")
ax3.set_xlabel(r"Distância de Dissimilaridade ($1 - |r|$)")
plt.tight_layout()
save_eda_fig(fig3, "25_agrupamento_hierarquico_dendrograma.pdf")
plt.show()

In [ ]:
# 7.2 Mapas Característicos de Performance Termodinâmica (3 figuras individuais)
sample_map = df_norm_valid.sample(n=min(30_000, len(df_norm_valid)), random_state=RANDOM_SEED)

# Plot 1: Curva Real vs. Esperada (Razão de Pressão x Head Politrópico)
fig1, ax1 = plt.subplots(figsize=(8.5, 5.0))
sc0 = ax1.scatter(
    sample_map["act_press_ratio"], sample_map["act_poly_head"],
    c=sample_map["act_poly_eff"], cmap="viridis", s=6, alpha=0.45, rasterized=True
)
exp_sorted = sample_map.sort_values("exp_press_ratio")
ax1.plot(exp_sorted["exp_press_ratio"], exp_sorted["exp_poly_head"], color="crimson", linewidth=2.0, label="Curva Esperada (Projeto)")
plt.colorbar(sc0, ax=ax1, label="Eficiência Real (%)")
ax1.set_title(r"Mapa $r_p \times H_p$ (Real vs. Curva de Projeto)")
ax1.set_xlabel("Razão de Compressão (-)")
ax1.set_ylabel("Head Politrópico (kJ/kg)")
ax1.legend(loc="lower right")
plt.tight_layout()
save_eda_fig(fig1, "26_mapa_termodinamico_rp_vs_hp.pdf")
plt.show()

# Plot 2: Potência Real vs. Esperada
fig2, ax2 = plt.subplots(figsize=(8.0, 5.0))
sc1 = ax2.scatter(
    sample_map["exp_shaft_power"], sample_map["act_shaft_power"],
    c=sample_map["shaft_pwr_dev"], cmap="plasma", s=6, alpha=0.45, rasterized=True
)
lims = [7000, 11000]
ax2.plot(lims, lims, "k--", linewidth=1.3, label="Paridade 1:1 (Act = Exp)")
plt.colorbar(sc1, ax=ax2, label="Desvio Potência (%)")
ax2.set_xlim(lims)
ax2.set_ylim(lims)
ax2.set_title("Potência Real vs. Esperada — Consumo Excedente (~+9.8%)")
ax2.set_xlabel("Potência Esperada (kW)")
ax2.set_ylabel("Potência Real (kW)")
ax2.legend(loc="upper left")
plt.tight_layout()
save_eda_fig(fig2, "27_mapa_termodinamico_potencia_paridade.pdf")
plt.show()

# Plot 3: Hexbin Acoplamento Head vs. Eficiência
fig3, ax3 = plt.subplots(figsize=(8.0, 5.0))
hb = ax3.hexbin(sample_map["poly_head_dev"], sample_map["poly_eff_dev"], gridsize=45, cmap="YlGnBu", mincnt=1)
plt.colorbar(hb, ax=ax3, label="Densidade de Pontos")
ax3.set_title("Acoplamento entre Desvio de Head e Desvio de Eficiência")
ax3.set_xlabel("Desvio de Head Politrópico (%)")
ax3.set_ylabel("Desvio de Eficiência Reconstruído (%)")
plt.tight_layout()
save_eda_fig(fig3, "28_mapa_termodinamico_head_vs_eff.pdf")
plt.show()

In [ ]:
# 7.3 PCA + Monitoramento Estatístico Multivariado (3 figuras individuais)
scaler_pca = StandardScaler()
X_norm_scaled = scaler_pca.fit_transform(df_norm_valid)

pca_full = PCA(random_state=RANDOM_SEED)
X_pca_scores = pca_full.fit_transform(X_norm_scaled)
exp_var = pca_full.explained_variance_ratio_
cum_var = np.cumsum(exp_var)

n_pcs = int(np.searchsorted(cum_var, 0.95) + 1)
print(f"Número de Componentes Principais para explicar >=95% da variância: {n_pcs} PCs ({cum_var[n_pcs-1]*100:.2f}%)")

scores_k = X_pca_scores[:, :n_pcs]
eigenvals_k = pca_full.explained_variance_[:n_pcs]
X_hat = scores_k @ pca_full.components_[:n_pcs]
hotelling_t2 = np.sum((scores_k ** 2) / eigenvals_k, axis=1)
spe_q = np.sum((X_norm_scaled - X_hat) ** 2, axis=1)

# Plot 1: Scree Plot da Variância Explicada
fig1, ax1 = plt.subplots(figsize=(8.0, 4.5))
pcs_idx = np.arange(1, len(exp_var) + 1)
ax1.bar(pcs_idx, exp_var * 100, alpha=0.75, color="#2b8cbe", label="Individual (%)")
ax1.plot(pcs_idx, cum_var * 100, marker="o", color="crimson", linewidth=2, label="Acumulada (%)")
ax1.axhline(95, color="gray", linestyle="--", linewidth=1, label="Limiar 95%")
ax1.set_title("Variância Explicada por Componente Principal (PCA)")
ax1.set_xlabel("Componente Principal")
ax1.set_ylabel("% da Variância")
ax1.legend()
plt.tight_layout()
save_eda_fig(fig1, "29_pca_scree_plot_variancia.pdf")
plt.show()

# Plot 2: Heatmap dos Loadings (Pesos) de PC1 e PC2
fig2, ax2 = plt.subplots(figsize=(6.5, 6.0))
loadings = pd.DataFrame(
    pca_full.components_[:2].T,
    index=[c.replace("_", " ") for c in SENSOR_COLS],
    columns=["PC1", "PC2"],
)
sns.heatmap(loadings, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax2, cbar=False)
ax2.set_title(f"Loadings de PC1 ({exp_var[0]*100:.1f}%) e PC2 ({exp_var[1]*100:.1f}%)")
plt.tight_layout()
save_eda_fig(fig2, "30_pca_loadings_heatmap.pdf")
plt.show()

# Plot 3: Trajetória Temporal no Espaço Latente (PC1 x PC2)
fig3, ax3 = plt.subplots(figsize=(8.5, 5.0))
idx_sub = np.linspace(0, len(X_pca_scores) - 1, 20_000, dtype=int)
time_num = mdates.date2num(df_norm_valid.index[idx_sub])
sc_pca = ax3.scatter(
    X_pca_scores[idx_sub, 0], X_pca_scores[idx_sub, 1],
    c=time_num, cmap="Spectral", s=5, alpha=0.45, rasterized=True
)
cbar = plt.colorbar(sc_pca, ax=ax3)
cbar.ax.yaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax3.set_title("Trajetória Temporal no Espaço Latente (PC1 × PC2)")
ax3.set_xlabel("PC1 (Desvio Global / Carga Esperada)")
ax3.set_ylabel("PC2 (Nível de Potência e Pressão Real)")
plt.tight_layout()
save_eda_fig(fig3, "31_pca_projecao_latente_tempo.pdf")
plt.show()

In [ ]:
# 7.4 Identificação Não Supervisionada de Regimes Operacionais (K-Means)
sample_idx_sil = np.random.choice(len(X_norm_scaled), size=min(8_000, len(X_norm_scaled)), replace=False)
sil_scores = {}
for k_cand in [2, 3, 4, 5]:
    km_tmp = KMeans(n_clusters=k_cand, n_init=10, random_state=RANDOM_SEED)
    lbl_tmp = km_tmp.fit_predict(X_norm_scaled[sample_idx_sil])
    sil_scores[k_cand] = silhouette_score(X_norm_scaled[sample_idx_sil], lbl_tmp)

print("Silhouette Score por número de regimes candidatos:", {k: round(v, 3) for k, v in sil_scores.items()})

N_REGIMES = 3
kmeans = KMeans(n_clusters=N_REGIMES, n_init=10, random_state=RANDOM_SEED)
raw_labels = kmeans.fit_predict(X_norm_scaled)

regime_order = (
    pd.Series(df_norm_valid["act_poly_eff"].values)
    .groupby(raw_labels)
    .mean()
    .sort_values(ascending=False)
    .index
)
label_remap = {old: f"Regime {new+1}" for new, old in enumerate(regime_order)}
regimes_series = pd.Series([label_remap[r] for r in raw_labels], index=df_norm_valid.index, name="Regime")

regime_profile = df_norm_valid.groupby(regimes_series)[[
    "act_poly_eff", "act_shaft_power", "act_press_ratio", "act_poly_head", "shaft_pwr_dev", "poly_eff_dev"
]].mean().round(2)
regime_profile["Participação (%)"] = (regimes_series.value_counts(normalize=True) * 100).round(2)
print("\nPerfil Termodinâmico Médio dos 3 Regimes Operacionais Descobertos:")
display(regime_profile)

# Plot: Evolução dos Regimes ao longo do tempo (PDF individual)
regime_1h = regimes_series.resample("1h").agg(lambda s: s.mode().iloc[0] if len(s) > 0 else np.nan).dropna()
eff_1h_aligned = df_1h.loc[regime_1h.index, "act_poly_eff"]

fig, ax = plt.subplots(figsize=(12, 4.5))
reg_colors = {"Regime 1": "#1b9e77", "Regime 2": "#377eb8", "Regime 3": "#e41a1c"}
for reg_name, cor in reg_colors.items():
    m = regime_1h == reg_name
    ax.scatter(eff_1h_aligned.index[m], eff_1h_aligned[m], s=8, color=cor, alpha=0.75, label=reg_name)

ax.set_title("Evolução dos Regimes Operacionais (K-Means Multivariado) sobre a Eficiência Politrópica (1h)")
ax.set_ylabel("Eficiência Politrópica Real (%)")
ax.set_xlabel("Data")
ax.legend(markerscale=2.5, loc="upper right")
plt.tight_layout()
save_eda_fig(fig, "32_regimes_operacionais_kmeans_tempo.pdf")
plt.show()

## 8. Detecção de Anomalias Multivariadas, Mudanças Estruturais (CUSUM) e Preparação para Deep Learning

Mesmo dentro da janela de `Operação Normal`, anomalias sutis de processo ocorrem quando:
- Uma variável isolada foge da sua faixa local (**Z-Score Robusto Móvel**);
- A combinação conjunta dos sensores viola a geometria normal do espaço multivariado (**`IsolationForest`** + **Erro de Reconstrução PCA-SPE**);
- Ocorre uma deriva persistente de eficiência (**CUSUM — Cumulative Sum Control Chart**).

In [ ]:
# 8.1 Detecção de Anomalias em figuras individuais (Consenso e PCA-SPE separados)
df_anom = df_5min.dropna().copy()
X_anom_scaled = RobustScaler().fit_transform(df_anom)

# (1) Z-Score Robusto Móvel na Eficiência Real
med_roll = df_anom["act_poly_eff"].rolling(288, center=True, min_periods=36).median()
mad_roll = (df_anom["act_poly_eff"] - med_roll).abs().rolling(288, center=True, min_periods=36).median() * 1.4826
z_local_eff = ((df_anom["act_poly_eff"] - med_roll) / mad_roll.replace(0, np.nan)).fillna(0.0)
flag_univar = z_local_eff.abs() > 4.0

# (2) Isolation Forest Multivariado
iso = IsolationForest(
    n_estimators=200,
    max_samples=2048,
    contamination=0.01,
    random_state=RANDOM_SEED,
    n_jobs=-1,
)
flag_isoforest = pd.Series(iso.fit_predict(X_anom_scaled) == -1, index=df_anom.index)

# (3) Erro de Reconstrução PCA (SPE / Q-statistic) > P99
X_anom_std = scaler_pca.transform(df_anom)
scores_anom = pca_full.transform(X_anom_std)[:, :n_pcs]
X_anom_hat = scores_anom @ pca_full.components_[:n_pcs]
spe_5min = pd.Series(np.sum((X_anom_std - X_anom_hat) ** 2, axis=1), index=df_anom.index)
spe_thr_99 = spe_5min.quantile(0.99)
flag_pca_spe = spe_5min > spe_thr_99

# Consenso Multivariado
consensus_votes = flag_univar.astype(int) + flag_isoforest.astype(int) + flag_pca_spe.astype(int)
flag_consensus = consensus_votes >= 2

# Plot 1: Anomalias de Consenso sobre a Eficiência
fig1, ax1 = plt.subplots(figsize=(12, 4.5))
ax1.plot(df_anom.index, df_anom["act_poly_eff"], color="#008080", linewidth=0.7, alpha=0.8, label="Eficiência Real (5 min)")
ax1.scatter(
    df_anom.index[flag_consensus],
    df_anom.loc[flag_consensus, "act_poly_eff"],
    color="crimson", s=18, zorder=5, label=f"Anomalias por Consenso (n={flag_consensus.sum()})"
)
ax1.set_title("Anomalias Multivariadas por Consenso Destacadas sobre a Eficiência Politrópica")
ax1.set_ylabel("Eficiência (%)")
ax1.set_xlabel("Data")
ax1.legend(loc="upper right")
plt.tight_layout()
save_eda_fig(fig1, "33_anomalias_consenso_eficiencia.pdf")
plt.show()

# Plot 2: Monitoramento PCA-SPE (Estatística Q) em Escala Log
fig2, ax2 = plt.subplots(figsize=(12, 4.2))
ax2.plot(spe_5min.index, spe_5min, color="#7570b3", linewidth=0.8, label="Estatística SPE / Q (Erro Reconstrução PCA)")
ax2.axhline(spe_thr_99, color="crimson", linestyle="--", linewidth=1.2, label=f"Limiar P99 ({spe_thr_99:.2f})")
ax2.set_yscale("log")
ax2.set_title("Monitoramento de Quebra de Correlação Física — PCA-SPE (Escala Log)")
ax2.set_ylabel("SPE / Q (log)")
ax2.set_xlabel("Data")
ax2.legend(loc="upper right")
plt.tight_layout()
save_eda_fig(fig2, "34_anomalias_pca_spe_q_statistic.pdf")
plt.show()

In [ ]:
# 8.2 Monitoramento de Deriva Estrutural (CUSUM) e Volatilidade em figuras individuais
eff_daily = df_1d["act_poly_eff"].dropna()
mu_ref = eff_daily.iloc[:14].mean()
sigma_ref = eff_daily.std()
k_slack = 0.25 * sigma_ref

cusum_neg = np.zeros(len(eff_daily))
cusum_pos = np.zeros(len(eff_daily))
vals_d = eff_daily.values
for t in range(1, len(vals_d)):
    cusum_pos[t] = max(0.0, cusum_pos[t - 1] + (vals_d[t] - mu_ref) - k_slack)
    cusum_neg[t] = max(0.0, cusum_neg[t - 1] - (vals_d[t] - mu_ref) - k_slack)

# Plot 1: Carta CUSUM sobre a Eficiência Diária
fig1, ax1 = plt.subplots(figsize=(11, 4.2))
ax1.plot(eff_daily.index, cusum_neg, color="crimson", linewidth=2.2, label=r"CUSUM Inferior ($C^-$: Acúmulo de Perda de Eficiência)")
ax1.plot(eff_daily.index, cusum_pos, color="seagreen", linewidth=1.6, label=r"CUSUM Superior ($C^+$)")
ax1.axvline(pd.Timestamp("2019-11-21"), color="navy", linestyle="--", label="Inflexão / Recuperação (~21/Nov/2019)")
ax1.set_title("Carta CUSUM sobre a Eficiência Diária (Referência = Primeiras 2 Semanas de Set/2019)")
ax1.set_ylabel("Soma Acumulada")
ax1.set_xlabel("Data")
ax1.legend(loc="upper left")
plt.tight_layout()
save_eda_fig(fig1, "35_cusum_deriva_estrutural_eficiencia.pdf")
plt.show()

# Plot 2: Volatilidade Operacional (Desvio Padrão Móvel 24h da Potência)
vol_24h = df_1h["act_shaft_power"].rolling(24, min_periods=6).std()
fig2, ax2 = plt.subplots(figsize=(11, 4.0))
ax2.plot(vol_24h.index, vol_24h, color="#d95f02", linewidth=1.1)
ax2.set_title("Volatilidade Operacional — Desvio Padrão Móvel de 24h da Potência de Eixo (kW)")
ax2.set_ylabel(r"$\sigma_{24h}$ Potência (kW)")
ax2.set_xlabel("Data")
plt.tight_layout()
save_eda_fig(fig2, "36_volatilidade_movel_potencia_24h.pdf")
plt.show()

## 9. Preparação Recomendada para Modelos de Deep Learning em Séries Temporais

A partir das descobertas empíricas obtidas ao longo deste notebook, qualquer arquitetura de *Deep Learning* aplicada a `compressor.csv` (**Forecasting Multivariado**, **Imputação Virtual de Sensores**, ou **Autoencoder de Detecção Precoce de Falhas CBM**) deve respeitar cinco diretrizes críticas:

1. **Regularização obrigatória na grade de `30s` (ou `5min`/`15min`)**: elimina os `~28.546` registros partidos por *clock skew* de 2 segundos sem perda de informação;
2. **Descarte de `poly_eff_dev_raw` (100% nula)** e substituição opcional por `poly_eff_dev` calculada analiticamente (ou remoção das colunas `*_dev` caso o objetivo seja evitar redundância algébrica entre `Act` e `Exp`);
3. **Filtragem de paradas e *glitches* de partida**: valores como `act_poly_eff = 5879%` ou `-742%` explodem gradientes de MSE/Huber se incluídos sem máscara de regime operacional;
4. **Divisão estritamente cronológica (Treino → Validação → Teste)** respeitando a continuidade dos blocos temporais (nunca *shuffle* aleatório de janelas sobrepostas!);
5. **Ajuste (`fit`) do normalizador exclusivamente no conjunto de treino** para evitar *data leakage*.

Abaixo demonstramos uma função enxuta e reutilizável de construção de tensores 3D `(N_janelas, seq_len, n_features)` pronta para PyTorch / TensorFlow / Keras.

In [ ]:
# 9.1 Pipeline Reprodutível Livre de Data Leakage para Deep Learning (Janelas Deslizantes 3D)
DL_FEATURES = [
    "act_press_ratio", "exp_press_ratio",
    "act_poly_head", "exp_poly_head",
    "act_shaft_power", "exp_shaft_power",
    "act_poly_eff", "exp_poly_eff",
]
TARGET_COL = "act_poly_eff"

# Usar série regularizada em 5 minutos (alta relação sinal/ruído, ~52.400 passos em 6 meses)
df_dl = df_5min[DL_FEATURES].copy()

# Interpolação apenas de micro-lacunas (até 15 min = 3 passos); lacunas maiores quebram janelas com segurança
df_dl = df_dl.interpolate(method="time", limit=3)

n_total = len(df_dl)
i_train = int(n_total * 0.70)
i_val = int(n_total * 0.85)

train_df = df_dl.iloc[:i_train]
val_df = df_dl.iloc[i_train:i_val]
test_df = df_dl.iloc[i_val:]

scaler_dl = StandardScaler()
scaler_dl.fit(train_df.dropna())  # FIT APENAS NO TREINO!

def build_clean_windows(frame: pd.DataFrame, scaler: StandardScaler, lookback: int = 72, horizon: int = 12, stride: int = 3):
    '''
    Constrói tensores X (N, lookback, F) e y (N, horizon) descartando automaticamente
    qualquer janela que atravesse paradas operacionais ou gaps de historiador (NaN).
    Com grade de 5 min: lookback=72 equivale a 6h de histórico; horizon=12 equivale a 1h à frente.
    '''
    arr = frame.values
    valid_row = ~np.isnan(arr).any(axis=1)
    arr_scaled = np.where(valid_row[:, None], scaler.transform(np.nan_to_num(arr)), np.nan)
    target_idx = frame.columns.get_loc(TARGET_COL)

    X_list, y_list, ts_list = [], [], []
    total_len = lookback + horizon
    for start in range(0, len(arr_scaled) - total_len + 1, stride):
        end_x = start + lookback
        end_y = end_x + horizon
        if valid_row[start:end_y].all():
            X_list.append(arr_scaled[start:end_x])
            y_list.append(arr_scaled[end_x:end_y, target_idx])
            ts_list.append(frame.index[end_x])

    return np.asarray(X_list, dtype=np.float32), np.asarray(y_list, dtype=np.float32), pd.DatetimeIndex(ts_list)

X_train, y_train, ts_train = build_clean_windows(train_df, scaler_dl, lookback=72, horizon=12, stride=3)
X_val, y_val, ts_val = build_clean_windows(val_df, scaler_dl, lookback=72, horizon=12, stride=3)
X_test, y_test, ts_test = build_clean_windows(test_df, scaler_dl, lookback=72, horizon=12, stride=3)

print("Tensores 3D prontos para Treinamento de Redes Neurais (LSTM / GRU / TCN / Transformer):")
print(f"  Treino    ({train_df.index.min().date()} -> {train_df.index.max().date()}): X={X_train.shape}, y={y_train.shape}")
print(f"  Validação ({val_df.index.min().date()} -> {val_df.index.max().date()}): X={X_val.shape}, y={y_val.shape}")
print(f"  Teste     ({test_df.index.min().date()} -> {test_df.index.max().date()}): X={X_test.shape}, y={y_test.shape}")

---

## 10. Síntese Executiva das Descobertas

1. **Estrutura e Cobertura**: O dataset abrange **6 meses completos** (`01/Set/2019` a `01/Mar/2020`, `547.508` registros brutos) de monitoramento CBM do 1º estágio de um compressor centrífugo, organizado em 4 grandezas físicas (*Razão de Compressão*, *Head Politrópico*, *Potência de Eixo* e *Eficiência Politrópica*) nas visões *Actual*, *Expected* e *Deviation*.
2. **Diagnóstico do Historiador (Split-Row Assíncrono)**: Os `~5,24%` de valores ausentes por coluna nos dados brutos **não são falhas de sensor**, mas sim um deslocamento de `2s` na gravação de subconjuntos de tags (`28s + 2s = 30s`). A reamostragem em grade regular de `30s` recupera **99,82% de completude** em todas as 11 tags ativas.
3. **Reconstrução Física de `Poly Eff Dev`**: A coluna `PH (CBM) 1st Stage Poly Eff Dev` está 100% vazia no CSV bruto, mas validamos analiticamente ($R^2 > 0,9999$, $\text{MAE} < 0,04\text{ p.p.}$) que o sistema CBM adota $\text{Dev} = \frac{\text{Act} - \text{Exp}}{\text{Exp}} \times 100$, permitindo sua reconstrução exata.
4. **Paradas, Compressão *Deadband* e *Glitches* de Partida**: Foram identificadas paradas operacionais relevantes (destaque para `12/Set`, `24–25/Set` com ~25,8h de duração e `10/Out/2019`), durante as quais o historiador reduz a taxa de gravação para intervalos de `600s`. Nas rampas de retorno à operação, a razão por potência próxima de zero gera *glitches* numéricos extremos (`-742%` a `+5.879%` de eficiência) que precisam ser mascarados antes de qualquer treino.
5. **Ciclos de Degradação e Recuperação**: As médias móveis de 7 dias, a decomposição STL e a carta CUSUM revelam dois ciclos claros: queda gradual da eficiência politrópica de `~86,1%` (início de Set/2019) até `~81,5%` (início de Nov/2019), seguida de recuperação no fim de Nov/2019 para `~84,9%` e nova deriva gradual até Fev/2020 (`~82,3%`), além de sazonalidade térmica diurna consistente (período de `24h` e harmônico de `12h` no espectro de Welch).
6. **Dimensionalidade Intrínseca e MSPC**: Devido ao forte acoplamento termodinâmico entre pressão, head, potência e eficiência, apenas **2 a 3 Componentes Principais explicam >95% da variância**, tornando estatísticas multivariadas como **PCA-SPE ($Q$)** e **Isolation Forest** altamente eficazes para detectar quebras sutis de coerência física entre sensores.